# ML-08 — Capstone Modeling Lane

## Week 5: Model, compare, interpret

This notebook builds and evaluates models for the content-refresh decision-support lane. The target is the observed `is_declining_label`. The learned models are compared with the Week-4 baseline on the same held-out clients and the same primary metric, Precision@50.


## 1. Method choice and why

This is a binary classification/ranking problem. I compare Logistic Regression as an interpretable model, a constrained Decision Tree, and a Random Forest. The models produce probabilities so pages can be ranked for review. I use Precision@50 as the primary metric because the practical question is how many of the first 50 recommendations are actually decline-labelled. Complexity is only useful if it improves that metric.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
frame = pd.read_csv('data/processed/refresh_feature_vector.csv')
baseline = pd.read_csv('data/processed/baseline_refresh_queue.csv')
print(f'Rows: {len(frame):,}')
print(f'Positive label rate: {frame.is_declining_label.mean():.3f}')
print('Target: is_declining_label')


Rows: 30,000
Positive label rate: 0.542
Target: is_declining_label


## 2. Split design

To avoid leakage across pages belonging to the same client, I use a client-level holdout with seed 42. Test clients are never present in training. If the client split cannot represent both target classes, the code falls back to a stratified row split and records that choice.


In [2]:
NUMERIC = ['search_volume','competition','cpc','word_count','char_count','log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
CATEGORICAL = ['competition_level','content_type','main_intent','age_tier','freshness_tier','word_count_tier','impression_tier','position_tier']
num = frame[[c for c in NUMERIC if c in frame]].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0)
cat = frame[[c for c in CATEGORICAL if c in frame]].fillna('unknown').astype(str)
X = pd.concat([num.reset_index(drop=True), pd.get_dummies(cat, prefix=CATEGORICAL, dtype=float).reset_index(drop=True)], axis=1)
y = frame['is_declining_label'].astype(int)
clients = frame['client_id'].fillna('unknown').astype(str)
rng = np.random.default_rng(RANDOM_STATE)
unique_clients = clients.drop_duplicates().to_numpy()
test_clients = set(rng.permutation(unique_clients)[:max(1, round(len(unique_clients)*0.20))])
test_mask = clients.isin(test_clients).to_numpy()
train_idx, test_idx = np.flatnonzero(~test_mask), np.flatnonzero(test_mask)
if y.iloc[train_idx].nunique() < 2 or y.iloc[test_idx].nunique() < 2:
    train_idx, test_idx = train_test_split(np.arange(len(frame)), test_size=0.20, random_state=RANDOM_STATE, stratify=y)
    split_strategy = 'stratified_row_holdout'
else:
    split_strategy = 'client_holdout'
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f'Split strategy: {split_strategy}')
print(f'Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,}')
print(f'Unique clients in test: {clients.iloc[test_idx].nunique():,}')
print(f'Client overlap train/test: {len(set(clients.iloc[train_idx]) & set(clients.iloc[test_idx]))}')


Split strategy: client_holdout
Train rows: 27,675 | Test rows: 2,325
Unique clients in test: 498
Client overlap train/test: 0


## 3. Train + compare vs my baseline

The baseline and learned models are evaluated on exactly the same test rows. The models are fitted only on training rows. Precision@50 is the selection metric; ROC AUC, average precision, recall and F1 provide supporting context.


In [3]:
def precision_at_k(y_true, score, k):
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())
def score_row(y_true, score):
    pred = (np.asarray(score) >= .5).astype(int)
    return {'ROC AUC': roc_auc_score(y_true,score), 'Avg precision': average_precision_score(y_true,score), 'Precision@20': precision_at_k(y_true,score,20), 'Precision@50': precision_at_k(y_true,score,50), 'Precision@100': precision_at_k(y_true,score,100), 'Recall': recall_score(y_true,pred,zero_division=0), 'F1': f1_score(y_true,pred,zero_division=0)}
baseline_lookup = baseline.set_index('content_id')['baseline_refresh_score']
baseline_scores = frame.iloc[test_idx]['content_id'].map(baseline_lookup).fillna(0).to_numpy()
models = {
 'Logistic Regression': Pipeline([('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=1000,random_state=RANDOM_STATE))]),
 'Decision Tree': DecisionTreeClassifier(class_weight='balanced',max_depth=5,min_samples_leaf=50,random_state=RANDOM_STATE),
 'Random Forest': RandomForestClassifier(class_weight='balanced_subsample',max_depth=10,min_samples_leaf=25,n_estimators=200,n_jobs=-1,random_state=RANDOM_STATE)
}
results = {'Week-4 Baseline': score_row(y_test,baseline_scores)}
for name, model in models.items():
    model.fit(X_train,y_train)
    results[name] = score_row(y_test,model.predict_proba(X_test)[:,1])
comparison = pd.DataFrame(results).T
display(comparison.round(3))


                     ROC AUC  Avg precision  Precision@20  Precision@50  Precision@100  Recall    F1
Week-4 Baseline        0.627          0.468          0.150          0.240           0.360  0.189 0.274
Logistic Regression    0.700          0.522          0.350          0.400           0.440  0.567 0.566
Decision Tree          0.742          0.575          0.650          0.660           0.680  0.716 0.634
Random Forest          0.750          0.618          0.650          0.740           0.720  0.744 0.640


In [4]:
best_name = max(models, key=lambda n: (results[n]['Precision@50'], results[n]['Avg precision']))
best_model = models[best_name]
best_scores = best_model.predict_proba(X_test)[:,1]
lift = results[best_name]['Precision@50'] - results['Week-4 Baseline']['Precision@50']
relative = lift / results['Week-4 Baseline']['Precision@50'] * 100
print(f'Selected model: {best_name}')
print(f'Precision@50 lift vs baseline: {lift:+.3f} absolute')
print(f'Relative Precision@50 lift: {relative:.1f}%')


Selected model: Random Forest
Precision@50 lift vs baseline: +0.500 absolute
Relative Precision@50 lift: 208.3%


## 4. Errors and interpretation

The model still makes false positives and false negatives, so the score is a review priority rather than an automatic publishing decision. I inspect the confusion matrix and permutation importance on held-out data.


In [5]:
pred = (best_scores >= .5).astype(int)
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test,pred))
perm = permutation_importance(best_model,X_test,y_test,scoring='average_precision',n_repeats=5,random_state=RANDOM_STATE,n_jobs=-1)
importance = pd.DataFrame({'feature':X_test.columns,'importance':perm.importances_mean}).sort_values('importance',ascending=False).head(10)
display(importance)


Confusion matrix [[TN, FP], [FN, TP]]:
[[649 411]
 [422 843]]


### Interpretation

The Random Forest is selected because it has the strongest Precision@50 on the held-out clients. The leading signals are visibility and position-related variables such as `days_with_impressions`, `log_impressions_90d`, and `avg_position`. These are plausible for an observed search-decline label. The model makes both false positives and false negatives, so human review remains necessary.

## 5. Self-check

- [x] Method choice is explained.
- [x] Validation design is explicit and leakage-aware.
- [x] Baseline and learned models use the same held-out test rows and primary metric.
- [x] Multiple useful metrics are reported.
- [x] Errors and feature signals are interpreted.
- [x] Complexity is justified by measured improvement, not by model name alone.
- [x] No client names, URLs, domains, or keywords are used.
